## Download Dataset

In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("appetukhov/international-trade-database")

print("Path to dataset files:", path)

Path to dataset files: C:\Users\Wyndpulse\.cache\kagglehub\datasets\appetukhov\international-trade-database\versions\1


In [2]:
import pandas as pd
pd.options.mode.copy_on_write = True

## Read Dataset
**Before performing below cell, move the dataset from the above path to current folder**

In [3]:
df = pd.read_csv('trade_1988_2021.csv')
df

,ReporterISO3,ReporterName,PartnerISO3,PartnerName,Year,TradeFlowName,TradeValue in 1000 USD
0,AFG,Afghanistan,SWE,Sweden,2017,Export,86.752
1,AFG,Afghanistan,JOR,Jordan,2018,Export,2796.481
2,AFG,Afghanistan,JOR,Jordan,2017,Export,3100.187
3,AFG,Afghanistan,ITA,Italy,2018,Export,279.918
4,AFG,Afghanistan,ITA,Italy,2017,Export,416.642
...,...,...,...,...,...,...,...
634504,ZWE,Zimbabwe,BRA,Brazil,2000,Export,1267.731
634505,ZWE,Zimbabwe,BOL,Bolivia,2000,Export,2.635
634506,ZWE,Zimbabwe,BMU,Bermuda,2002,Export,10.599
634507,ZWE,Zimbabwe,BLZ,Belize,2000,Export,17.772


In [4]:
df_2019 = df[df.Year == 2019]
df_2019.shape

(23009, 7)

## Creating Columns to join on
Want to join (Report, Partner) pair with (Partner, Reporter) pair

In [5]:
df_2019['OriginDestinationPair'] = df_2019['ReporterName'] + df_2019['PartnerName']
df_2019['DestinationOriginPair'] = df_2019['PartnerName'] + df_2019['ReporterName']

## Perform Join, and calculate net exports with pair

In [6]:
df_joined = df_2019.set_index('OriginDestinationPair').join(df_2019.set_index('DestinationOriginPair'), lsuffix='_origin', rsuffix='_destination', how='left')
df_joined['TradeValue in 1000 USD_destination'] = df_joined['TradeValue in 1000 USD_destination'].fillna(0)
df_joined['ExportMinusImport'] = df_joined['TradeValue in 1000 USD_origin'] - df_joined['TradeValue in 1000 USD_destination']
df_joined

,ReporterISO3_origin,ReporterName_origin,PartnerISO3_origin,PartnerName_origin,Year_origin,TradeFlowName_origin,TradeValue in 1000 USD_origin,DestinationOriginPair,ReporterISO3_destination,ReporterName_destination,PartnerISO3_destination,PartnerName_destination,Year_destination,TradeFlowName_destination,TradeValue in 1000 USD_destination,OriginDestinationPair,ExportMinusImport
OriginDestinationPair,,,,,,,,,,,,,,,,,
AfghanistanCentral African Republic,AFG,Afghanistan,CAF,Central African Republic,2019,Export,136.220,Central African RepublicAfghanistan,NaN,NaN,NaN,NaN,NaN,NaN,0.000,NaN,136.220
AfghanistanCanada,AFG,Afghanistan,CAN,Canada,2019,Export,1377.671,CanadaAfghanistan,CAN,Canada,AFG,Afghanistan,2019.0,Export,8085.790,CanadaAfghanistan,-6708.119
AfghanistanSwitzerland,AFG,Afghanistan,CHE,Switzerland,2019,Export,10.821,SwitzerlandAfghanistan,CHE,Switzerland,AFG,Afghanistan,2019.0,Export,4484.568,SwitzerlandAfghanistan,-4473.747
AfghanistanChina,AFG,Afghanistan,CHN,China,2019,Export,31001.902,ChinaAfghanistan,CHN,China,AFG,Afghanistan,2019.0,Export,600997.504,ChinaAfghanistan,-569995.602
AfghanistanCzech Republic,AFG,Afghanistan,CZE,Czech Republic,2019,Export,1.145,Czech RepublicAfghanistan,CZE,Czech Republic,AFG,Afghanistan,2019.0,Export,31173.226,Czech RepublicAfghanistan,-31172.081
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ZimbabweBrazil,ZWE,Zimbabwe,BRA,Brazil,2019,Export,1.009,BrazilZimbabwe,BRA,Brazil,ZWE,Zimbabwe,2019.0,Export,3805.057,BrazilZimbabwe,-3804.048
ZimbabweBulgaria,ZWE,Zimbabwe,BGR,Bulgaria,2019,Export,24.815,BulgariaZimbabwe,BGR,Bulgaria,ZWE,Zimbabwe,2019.0,Export,406.944,BulgariaZimbabwe,-382.129
ZimbabweBangladesh,ZWE,Zimbabwe,BGD,Bangladesh,2019,Export,0.004,BangladeshZimbabwe,NaN,NaN,NaN,NaN,NaN,NaN,0.000,NaN,0.004


## Take only entries where net exports is non-negative

In [7]:
df_cleaned = df_joined[df_joined.ExportMinusImport >= 0]
df_cleaned

,ReporterISO3_origin,ReporterName_origin,PartnerISO3_origin,PartnerName_origin,Year_origin,TradeFlowName_origin,TradeValue in 1000 USD_origin,DestinationOriginPair,ReporterISO3_destination,ReporterName_destination,PartnerISO3_destination,PartnerName_destination,Year_destination,TradeFlowName_destination,TradeValue in 1000 USD_destination,OriginDestinationPair,ExportMinusImport
OriginDestinationPair,,,,,,,,,,,,,,,,,
AfghanistanCentral African Republic,AFG,Afghanistan,CAF,Central African Republic,2019,Export,136.220,Central African RepublicAfghanistan,NaN,NaN,NaN,NaN,NaN,NaN,0.000,NaN,136.220
"AfghanistanIran, Islamic Rep.",AFG,Afghanistan,IRN,"Iran, Islamic Rep.",2019,Export,14619.800,"Iran, Islamic Rep.Afghanistan",NaN,NaN,NaN,NaN,NaN,NaN,0.000,NaN,14619.800
AfghanistanIraq,AFG,Afghanistan,IRQ,Iraq,2019,Export,14500.217,IraqAfghanistan,NaN,NaN,NaN,NaN,NaN,NaN,0.000,NaN,14500.217
AfghanistanJordan,AFG,Afghanistan,JOR,Jordan,2019,Export,1415.175,JordanAfghanistan,JOR,Jordan,AFG,Afghanistan,2019.0,Export,1114.071,JordanAfghanistan,301.104
AfghanistanKuwait,AFG,Afghanistan,KWT,Kuwait,2019,Export,309.687,KuwaitAfghanistan,KWT,Kuwait,AFG,Afghanistan,2019.0,Export,32.730,KuwaitAfghanistan,276.957
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ZimbabweBotswana,ZWE,Zimbabwe,BWA,Botswana,2019,Export,43383.983,BotswanaZimbabwe,BWA,Botswana,ZWE,Zimbabwe,2019.0,Export,33978.608,BotswanaZimbabwe,9405.375
ZimbabweBarbados,ZWE,Zimbabwe,BRB,Barbados,2019,Export,11.637,BarbadosZimbabwe,BRB,Barbados,ZWE,Zimbabwe,2019.0,Export,9.650,BarbadosZimbabwe,1.987
ZimbabweBangladesh,ZWE,Zimbabwe,BGD,Bangladesh,2019,Export,0.004,BangladeshZimbabwe,NaN,NaN,NaN,NaN,NaN,NaN,0.000,NaN,0.004


## Some column name clean-up, and then save to file

In [8]:
df_final = df_cleaned.rename(columns={'ReporterName_origin': 'Origin', 'PartnerName_origin': 'Destination', 'ExportMinusImport': 'NetExports', 'Year_origin': 'Year'})
df_final[['Origin', 'Destination', 'NetExports']]

,Origin,Destination,NetExports
OriginDestinationPair,,,
AfghanistanCentral African Republic,Afghanistan,Central African Republic,136.220
"AfghanistanIran, Islamic Rep.",Afghanistan,"Iran, Islamic Rep.",14619.800
AfghanistanIraq,Afghanistan,Iraq,14500.217
AfghanistanJordan,Afghanistan,Jordan,301.104
AfghanistanKuwait,Afghanistan,Kuwait,276.957
...,...,...,...
ZimbabweBotswana,Zimbabwe,Botswana,9405.375
ZimbabweBarbados,Zimbabwe,Barbados,1.987
ZimbabweBangladesh,Zimbabwe,Bangladesh,0.004


In [14]:
df_final[['Origin', 'Destination', 'NetExports']].reset_index(drop=True).to_csv('2019_net_trade_data.csv')